In [22]:
import ollama
import json
from pydantic import BaseModel
from typing import List, Optional

import subprocess
import cv2
import numpy as np
import img2pdf
from pdf2image import convert_from_path
import pymupdf4llm
import os

# Crear carpetas temporales para el pipeline
os.makedirs("temp", exist_ok=True)


importante! hay que instalar OCRmyPDF

sudo apt update

sudo apt install ocrmypdf tesseract-ocr-spa ghostscript unpaper -y

También hay que instalar poppler

sudo apt update && sudo apt install poppler-utils -y

In [61]:
import os
import stat
import subprocess
from PIL import Image

def limpiar_y_enderezar(ruta_imagen, output_pdf_path="temp/limpio.pdf"):
    print(f"1. Preparando imagen y ejecutando OCRmyPDF...")
    
    # --- INICIO DEL PREPROCESAMIENTO (Normalización de Transparencias/PNG) ---
    img_segura = "temp/input_seguro.jpg"
    img = Image.open(ruta_imagen)
    
    # Chequeamos si la imagen tiene canal alfa (RGBA, LA) o paleta con transparencia
    if img.mode in ('RGBA', 'LA') or (img.mode == 'P' and 'transparency' in img.info):
        print("   [!] Canal alfa detectado. Normalizando a RGB puro con fondo blanco...")
        # Creamos un lienzo blanco del mismo tamaño
        fondo = Image.new("RGB", img.size, (255, 255, 255))
        # Pegamos la imagen original usando su propio canal de transparencia como máscara
        if img.mode == 'RGBA':
            fondo.paste(img, mask=img.split()[3])
        else:
            fondo.paste(img.convert('RGBA'), mask=img.convert('RGBA').split()[3])
        
        fondo.save(img_segura, "JPEG", quality=100)
        ruta_imagen = img_segura  # OCRmyPDF ahora usará esta versión segura
        
    elif img.mode != 'RGB' or ruta_imagen.lower().endswith('.png'):
        print(f"   [!] Normalizando formato de imagen a JPG estandar...")
        img.convert('RGB').save(img_segura, "JPEG", quality=100)
        ruta_imagen = img_segura
    # --- FIN DEL PREPROCESAMIENTO ---

    # --- INICIO DEL HACK: Crear un 'jbig2' falso ---
    fake_jbig2_path = "temp/jbig2"
    with open(fake_jbig2_path, "w") as f:
        f.write("#!/bin/sh\n")
        f.write("echo 'jbig2enc 0.29'\n") 
        
    os.chmod(fake_jbig2_path, os.stat(fake_jbig2_path).st_mode | stat.S_IEXEC)
    
    my_env = os.environ.copy()
    my_env["PATH"] = os.path.abspath("temp") + os.pathsep + my_env.get("PATH", "")
    # --- FIN DEL HACK ---
    
    comando = [
        "ocrmypdf",
        "--image-dpi", "300", 
        "--optimize", "0",    
        "--deskew",           
        "--clean",            
        "--force-ocr",        
        "--language", "spa",  
        ruta_imagen,          # <-- Ahora pasamos la imagen normalizada
        output_pdf_path       
    ]
    
    try:
        subprocess.run(comando, env=my_env, capture_output=True, text=True, check=True)
        print("✅ PDF limpio y enderezado con éxito.")
        return output_pdf_path
    except subprocess.CalledProcessError as e:
        print(f"❌ Error en OCRmyPDF (Código {e.returncode}):")
        print(f"--- DETALLE DEL ERROR ---\n{e.stderr}")
        return None

In [ ]:
# # FUNCIONES DE PROCESAMIENTO DE IMÁGENES
# # OCRmyPDF

# def limpiar_y_enderezar(jpg_path, output_pdf_path="temp/limpio.pdf"):
#     print(f"1. Ejecutando OCRmyPDF directamente sobre la imagen...")
    
#     comando = [
#         "ocrmypdf",
#         "--image-dpi", "300", 
#         "--optimize", "0",    # 🚀 CLAVE: Apagamos la compresión para evitar el error 'jbig2' y ahorrar CPU
#         "--deskew",           
#         "--clean",            
#         "--force-ocr",        
#         "--language", "spa",  
#         jpg_path,             
#         output_pdf_path       
#     ]
    
#     try:
#         subprocess.run(comando, capture_output=True, text=True, check=True)
#         print("✅ PDF limpio y enderezado con éxito.")
#         return output_pdf_path
#     except subprocess.CalledProcessError as e:
#         print(f"❌ Error en OCRmyPDF (Código {e.returncode}):")
#         print(f"--- DETALLE DEL ERROR ---\n{e.stderr}")
#         return None

In [65]:
from PIL import Image

def preparar_documento(pdf_limpio_path):
    print("3. Convirtiendo PDF limpio a imagen optimizada para IA...")
    # Bajamos los DPI a 100 de base para trabajar con una imagen más dócil
    paginas = convert_from_path(pdf_limpio_path, dpi=100)
    imagen = paginas[0]
    
    w, h = imagen.size
    
    # --- EL FIX DEFINITIVO (Bypass al auto-resize de Ollama) ---
    max_edge = 1008  # Límite ultra seguro (36 x 28)
    
    # Calculamos la proporción base para no deformar la hoja
    if w > h:
        new_w = max_edge
        new_h = int(h * (max_edge / w))
    else:
        new_h = max_edge
        new_w = int(w * (max_edge / h))
        
    # Aplicamos la regla estricta de múltiplos de 28 a la miniatura
    final_w = (new_w // 28) * 28
    final_h = (new_h // 28) * 28
    
    # Un "safety net" por si las dudas
    final_w = max(28, final_w)
    final_h = max(28, final_h)
    
    # Redimensionamiento final de alta calidad
    imagen = imagen.resize((final_w, final_h), Image.Resampling.LANCZOS)
    # --- FIN DEL FIX ---
    
    ruta_full = "temp/documento_completo.jpg"
    imagen.save(ruta_full, "JPEG", quality=95)
    
    print("4. Extrayendo esqueleto Markdown (PyMuPDF4LLM)...")
    markdown_structure = pymupdf4llm.to_markdown(pdf_limpio_path)
    
    return ruta_full, markdown_structure

In [39]:
# BLOQUE DE DEFINICIÓN DE ESQUEMA PARA LOS DATOS EXTRAÍDOS 
# PYDANTIC

# --- DEFINICIÓN DEL ESQUEMA DIRECTAMENTE AQUÍ ---
class StaffRow(BaseModel):
    categoria: str
    masculino: Optional[int] = 0
    femenino: Optional[int] = 0
    total: Optional[int] = 0

class PopulationRow(BaseModel):
    situacion_legal: str
    provincial_masc: Optional[int] = 0
    provincial_fem: Optional[int] = 0
    nacional_masc: Optional[int] = 0
    nacional_fem: Optional[int] = 0
    federal_masc: Optional[int] = 0
    federal_fem: Optional[int] = 0
    total: Optional[int] = 0

class SneepPage1(BaseModel):
    provincia: Optional[str] = None
    reparticion: Optional[str] = None
    nombre_establecimiento: Optional[str] = None
    tipo_establecimiento: Optional[str] = None
    domicilio_cp: Optional[str] = None
    telefono_fax: Optional[str] = None
    email: Optional[str] = None
    responsable_estadistica: Optional[str] = None
    capacidad_fisica_alojamiento: Optional[int] = 0
    alojados_celdas_individuales: Optional[int] = 0
    alojados_locales_colectivos: Optional[int] = 0
    dotacion_personal: List[StaffRow]
    poblacion_por_jurisdiccion: List[PopulationRow]

In [66]:
def ejecutar_pipeline_completo(jpg_original):
    # 1. Preprocesamiento (Limpiar y enderezar)
    pdf_limpio = limpiar_y_enderezar(jpg_original)
    if not pdf_limpio: return None
    
    # 2. Preparamos 1 sola imagen y el texto base
    ruta_full, md_struct = preparar_documento(pdf_limpio)
    
    # 3. Generamos el esquema JSON a partir de tu clase Pydantic
    esquema_json = json.dumps(SneepPage1.model_json_schema(), indent=2)
    
    # 4. Inferencia Híbrida
    SYSTEM_PROMPT = f"""
    Eres un experto en digitalización de formularios estadísticos penitenciarios (SNEEP). 
    Analiza la imagen completa del formulario.
    
    Estructura OCR de guía:
    {str(md_struct)[:1000]}
    
    REGLAS ESTRICTAS:
    1. Transcribe el texto manuscrito con exactitud. Si está vacío o ilegible, usa null.
    2. Para TODOS los campos numéricos, si están vacíos o tienen una raya, DEBES devolver el número 0. NUNCA devuelvas "".
    3. Devuelve EXCLUSIVAMENTE un objeto JSON que respete las llaves de este esquema EXACTO:
    {esquema_json}
    """
    
    print("5. Iniciando inferencia en Qwen2.5-VL (Imagen Única)...")
    response = ollama.chat(
        model='qwen2.5vl:7b',
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {
                'role': 'user', 
                'content': 'Procesa esta imagen y extrae los datos al JSON.',
                'images': [ruta_full] # <-- ¡CLAVE! Solo 1 imagen
            }
        ]
    )

    raw_json = response['message']['content']
    
    # Limpiamos los backticks de markdown por si la IA los agrega
    if "```json" in raw_json:
        raw_json = raw_json.split("```json")[1].split("```")[0].strip()
    elif "```" in raw_json:
        raw_json = raw_json.split("```")[1].strip()

    print("6. Validando con Pydantic...")
    try:
        data_validada = SneepPage1.model_validate_json(raw_json)
        print("✅ ¡Validación Pydantic superada!")
        return data_validada.model_dump()
    except Exception as e:
        print(f"❌ Error de validación Pydantic:\n{e}")
        print(f"\n--- JSON DEVUELTO POR LA IA ---\n{raw_json}")
        return None

In [68]:
resultado_final = ejecutar_pipeline_completo('cuestionario_sneep2_p1 - copia3.png')
if isinstance(resultado_final, dict):
    print(json.dumps(resultado_final, indent=2, ensure_ascii=False))

1. Preparando imagen y ejecutando OCRmyPDF...
   [!] Canal alfa detectado. Normalizando a RGB puro con fondo blanco...
✅ PDF limpio y enderezado con éxito.
3. Convirtiendo PDF limpio a imagen optimizada para IA...
4. Extrayendo esqueleto Markdown (PyMuPDF4LLM)...
=== Document parser messages ===
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            Using Tesseract for OCR processing.
OCR on page.number=0/1.

5. Iniciando inferencia en Qwen2.5-VL (Imagen Única)...
6. Validando con Pydantic...
✅ ¡Validación Pyd